# Intermediate Precipitation Processing

After running prec_daily_calcs.ipynb, this notebook filters out the daily watershed-mean precipitation by AR days and stores it in a large pickled dictionary.

In [33]:
import xarray as xa
import matplotlib.pyplot as plt
import numpy as np
import sys
sys.path.append("..")
import fmap as fm
import regionmask
import cftime
import geopandas as gpd
import pandas as pd
from glob import glob
import pickle

In [2]:
wusd3 = pd.read_csv('../wusd3_gcms.csv', index_col=0)
for i in range(1011, 1201, 20):
    wusd3.loc[len(wusd3)] = ['cesm2-le', 'CESM2-LE', str(i), '365_day']

In [3]:
# Build the dictionary structure.

watersheds = ['1503', '1605', '1606', '1701', '1702', '1703', '1705', '1706', '1707', '1708', '1709', '1710', '1711', '1712', '1801', '1802', '1803', '1804', '1805', '1806', '1807', 
              '1808', '1809', '1810']

ensembles = ["cmip6nonbc", "cmip6bc", "cesm2le", "wrfera5", "prism"]

ARDT = ["G18", "TE", "tARget"]

time_periods = ["historical", "near-term", "end-of-century"]

prec_AR_days = {
                ws: {
                    ens: {
                        ardt: {
                            tp: []  # variable-length, fill manually
                            for tp in time_periods
                        }
                        for ardt in ARDT
                    }
                    for ens in ensembles
                }
                for ws in watersheds
            }

In [22]:
# Iteratively get each model/watershed/ARDT/time period configuration and store the AR Day precipitation values as an array for each configuration.
for ws in watersheds:
    for ens in ensembles:
        for ardt in ARDT:
            for tp in time_periods:
                print(f"{ws:5}/{ens:20}/{ardt:6}/{tp:20}", end='\r')
                if ens in ['wrfera5', 'prism'] and tp in ['near-term', 'end-of-century']:
                    continue
                elif ens in ['cmip6nonbc', 'cmip6bc']:
                    use_ens = wusd3.iloc[[0,1,2,3,4,5,7,8,9,10,11,12,13,14,15]]
                elif ens=='cesm2le':
                    use_ens = wusd3.iloc[16:]
                elif ens in ['wrfera5', 'prism']:
                    use_ens = wusd3.iloc[[6]]
                value_array = []
                
                for row in use_ens.index:
                    model = use_ens.loc[row, 'Model']
                    member = use_ens.loc[row, 'Member']
                    calendar = use_ens.loc[row, 'Calendar']
                    syear = 1980 if tp=='historical' else 2016 if tp=='near-term' else 2066 if tp=='end-of-century' else -9999
                    eyear = 2014 if tp=='historical' else 2050 if tp=='near-term' else 2100 if tp=='end-of-century' else -9999
                    sdate = cftime.datetime(syear, 9, 1, calendar=calendar) if ens != 'prism' else cftime.datetime(1981, 1, 1, calendar='standard')
                    edate = cftime.datetime(eyear, 8, 31, calendar=calendar) if calendar != '360_day' else cftime.datetime(eyear, 8, 30, calendar=calendar)
                    
                    ARs = xa.open_dataset(f'/glade/work/tcorrie/ARdata/ARwatershed/{ardt}/ARwshed.{ardt}.{model}.{member}.nc', use_cftime=True)
                    AR_times = ARs.sel(time=slice(sdate,edate)).where(ARs.watershed==ws,drop=True).where(ARs.ARwshed == 1, drop=True).where((ARs.time.dt.month >= 10) | (ARs.time.dt.month <= 3), drop=True).time.values
                    if ens != 'prism':
                        prec = xa.open_dataset(f'/glade/work/tcorrie/ARdata/pmean/prec_mean.{ens}.{model}.{member}.nc')
                    else:
                        prec = xa.open_dataset(f'/glade/work/tcorrie/ARdata/pmean/prec_mean.prism.prism.nan.nc')
                    AR_day_mean_precs = prec.sel(time=AR_times).where(prec.region == ws, drop=True).dropna(dim='time').prec_mean.values
                    value_array.extend(AR_day_mean_precs.flatten().tolist())
                prec_AR_days[ws][ens][ardt][tp] = value_array

In [34]:
pickle.dump(prec_AR_days, open('/glade/work/tcorrie/ARdata/pmean/prec_mean_all.pkl', 'wb'))